# Association Rule

Dữ liệu giao thông có số lượng "Items" (các tuyến đường) rất lớn. Do đó, thay vì thuật toán Apriori truyền thống, nhóm ưu tiên sử dụng FP-Growth vì tốc độ tính toán ma trận thưa (sparse matrix) của nó vượt trội hơn.

### Bước 1: Định nghĩa Giỏ hàng (Transaction) và Mặt hàng (Item)

Chỉ tập trung vào các tình trạng giao thông xấu để tìm quy luật kẹt xe.



In [106]:
!pip install --upgrade jupyter_client

In [107]:
from datetime import datetime, timezone
now = datetime.now(timezone.utc)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import unicodedata

# 1. Lọc các dòng có mức độ kẹt xe cao (LOS là E hoặc F)
df_congestion = df_train_clean[df_train_clean['LOS'].isin(['E', 'F'])].copy()
df_weekday_congestion = df_congestion[df_congestion['is_weekend'] == False].copy()

# 2. Đồng bộ unicode Tiếng Việt (Đưa tất cả về chuẩn NFC)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].apply(lambda x: unicodedata.normalize('NFC', str(x)))

# 3. Làm sạch text (tránh trùng lặp tên đường)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.replace(r'\s+', ' ', regex=True)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.replace(r'^Đường\s+', '', regex=True)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.strip()

# 4. Gom giỏ hàng
transactions = df_weekday_congestion.groupby(['date', 'time_slot'])['street_name'].apply(lambda x: list(set(x))).tolist()
transactions = [t for t in transactions if len(t) > 1]
print(f"Số lượng Transactions (Khung giờ kẹt nhiều đường): {len(transactions):,}")
print(f"Mẫu 3 Transactions (Đã lọc trùng): {transactions[:3]}")

Số lượng Transactions (Khung giờ kẹt nhiều đường): 272
Mẫu 3 Transactions (Đã lọc trùng): [['Song Hành Xa lộ Hà Nội', 'Xa Lộ Hà Nội'], ['Xa Lộ Hà Nội', 'Quốc lộ 1'], ['Xa Lộ Hà Nội', 'Mai Chí Thọ']]


### Bước 2: Chạy FP-Growth và Trích xuất Luật (Rules)

Thuật toán AR yêu cầu đầu vào là một ma trận Boolean (True/False).

Việc thiết lập các tham số min_support và min_threshold cần được tinh chỉnh (fine-tune) dựa trên kích thước thực tế của tập dữ liệu.

In [109]:
# Huấn luyện FP-GROWTH
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

# min_support = 0.04 (phải xuất hiện cùng nhau ít nhất ~14 lần trên tổng 348 khung giờ)
# max_len = 3 (Chỉ tìm các tổ hợp gồm tối đa 3 tuyến đường để tránh bùng nổ tổ hợp)
frequent_itemsets = fpgrowth(df_transactions, min_support=0.04, use_colnames=True, max_len=3)
print(f"Tổ hợp phổ biến tìm được (min_support >= 0.04): {len(frequent_itemsets):,}")

if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

    # Lọc luật: Cần độ tin cậy >= 0.5
    strong_rules = rules[rules['confidence'] >= 0.5].copy()

    # Chỉ lấy các luật có đúng 1 con đường nguyên nhân (Antecedents độ dài = 1)
    strong_rules['antecedent_len'] = strong_rules['antecedents'].apply(lambda x: len(x))
    strong_rules = strong_rules[strong_rules['antecedent_len'] == 1]

    # Sắp xếp theo Support (Mức độ phổ biến) sau đó mới đến Lift (Độ liên kết)
    strong_rules = strong_rules.sort_values(by=['support', 'lift'], ascending=[False, False])

    print(f"Số lượng quy luật mạnh thực tế sau khi lọc: {len(strong_rules):,}")

    if len(strong_rules) > 0:
        strong_rules['antecedents_str'] = strong_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
        strong_rules['consequents_str'] = strong_rules['consequents'].apply(lambda x: ', '.join(list(x)))
        final_rules = strong_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]

        print("\nTop chuỗi ùn tắt có ý nghĩa cao nhất:")
        print(final_rules.head(10).to_string(index=False))

        final_rules.to_csv('traffic_rules_optimized.csv', index=False)

        # Lưu file
        final_rules.to_csv('traffic_association_rules_weekday.csv', index=False)
        print("\nĐã lưu kết quả xuất sắc nhất ra file: 'traffic_association_rules_weekday.csv'")
    else:
        print("\nKhông có luật nào đạt ngưỡng Confidence >= 0.4. Hãy thử giảm min_support xuống 0.005 hoặc giảm confidence xuống 0.3")
else:
    print("\nKhông tìm thấy tổ hợp phổ biến nào. Tập dữ liệu quá phân tán, hãy giảm min_support xuống 0.005")

Tổ hợp phổ biến tìm được (min_support >= 0.04): 63
Số lượng quy luật mạnh thực tế sau khi lọc: 35

Top chuỗi ùn tắt có ý nghĩa cao nhất:
       antecedents_str   consequents_str  support  confidence      lift
         Tô Hiến Thành    Lý Thường Kiệt 0.139706    0.808511  2.114566
     Nguyễn Ðình Chiểu       Trương Định 0.095588    0.553191  4.702128
           Trương Định Nguyễn Ðình Chiểu 0.095588    0.812500  4.702128
Hẻm 373 Lý Thường Kiệt    Lý Thường Kiệt 0.080882    0.880000  2.301538
               Lê Khôi    Lý Thường Kiệt 0.080882    0.880000  2.301538
         Lạc Long Quân    Lý Thường Kiệt 0.058824    0.941176  2.461538
         Hoàng Văn Thụ    Lý Thường Kiệt 0.058824    0.640000  1.673846
         Hoàng Văn Thụ      Trường Chinh 0.055147    0.600000  7.418182
          Trường Chinh     Hoàng Văn Thụ 0.055147    0.681818  7.418182
              Cộng Hòa      Trường Chinh 0.051471    0.875000 10.818182

Đã lưu kết quả xuất sắc nhất ra file: 'traffic_association_rules_weekd

### Nhận xét:

*   Nhận diện chính xác các "nút thắt" (Choke points) trọng yếu: Thuật toán đã tự động trích xuất được các chuỗi ùn tắc mang tính dây chuyền tại các nút giao huyết mạch. Điển hình là cụm Cộng Hòa -> Trường Chinh (độ tin cậy 87.5%, hệ số Lift đột biến 10.8) và Hoàng Văn Thụ <-> Trường Chinh (Lift 7.4), minh chứng rõ rệt cho sự sụp đổ năng lực thông hành có tính nhân quả tại khu vực cửa ngõ. Tương tự với cụm giao cắt Nguyễn Đình Chiểu <-> Trương Định (Lift 4.7).

*   Phát hiện hiệu ứng tràn bờ (Spillover Effect): Kết quả cho thấy rõ hiện tượng phương tiện từ các đường nhánh/hẻm đổ ra làm quá tải trục đường chính. Các tuyến như Hẻm 373 Lý Thường Kiệt, Lê Khôi, Lạc Long Quân khi ùn tắc có xác suất kéo theo sự tê liệt của toàn tuyến Lý Thường Kiệt lên tới 88% - 94% (Confidence).

*   Giá trị ứng dụng: Bộ 35 quy luật này cung cấp các bằng chứng thực nghiệm chặt chẽ về sự liên kết không gian của mạng lưới đường bộ. Đây là cơ sở dữ liệu quan trọng để xác định các vị trí cần ưu tiên can thiệp hạ tầng, định tuyến lại mạng lưới xe buýt, hoặc làm nền tảng đánh giá vị trí các điểm nút (nodes) trong quy hoạch phát triển đô thị.